# 07 — Prediction & Mapping
**Steps 27–29:** Apply trained model to annual datacubes (2018–2025), reconstruct spatial rasters, and export classified landcover maps.

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import joblib
from tqdm import tqdm
from torch_geometric.nn import GATConv

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
RAW_DIR       = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
EXPORTS_DIR   = os.path.join('..', 'data', 'exports')
OUTPUTS_DIR   = os.path.join('..', 'outputs')
os.makedirs(EXPORTS_DIR, exist_ok=True)

YEARS       = list(range(2018, 2026))
CLASS_NAMES = ['Water', 'Trees', 'Crops', 'Built-up Areas', 'Bare Ground']
# Colour map for maps
CLASS_COLORS = ['#4472C4', '#70AD47', '#FFC000', '#FF0000', '#A6A6A6']
N_CLASSES    = 5

In [ ]:
# ── Load model & scaler ───────────────────────────────────────
class ModalityBranch(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class FusionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(hidden_dim * 2, 2), nn.Softmax(dim=-1))
        self.proj = nn.Linear(hidden_dim, hidden_dim)
    def forward(self, o, r):
        w = self.attn(torch.cat([o, r], dim=-1))
        return F.relu(self.proj(w[:, 0:1] * o + w[:, 1:2] * r))

class CMAGNN(nn.Module):
    def __init__(self, n_optical=5, n_radar=3, hidden_dim=64, n_classes=5, dropout=0.3, gat_heads=4):
        super().__init__()
        self.optical_branch = ModalityBranch(n_optical, hidden_dim, dropout)
        self.radar_branch   = ModalityBranch(n_radar, hidden_dim, dropout)
        self.fusion         = FusionLayer(hidden_dim)
        self.gat1 = GATConv(hidden_dim, hidden_dim // gat_heads, heads=gat_heads, dropout=dropout, concat=True)
        self.gat2 = GATConv(hidden_dim, hidden_dim, heads=1, dropout=dropout, concat=False)
        self.bn1  = nn.BatchNorm1d(hidden_dim); self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)
    def forward(self, x, edge_index):
        h = self.fusion(self.optical_branch(x[:, :5]), self.radar_branch(x[:, 5:]))
        h = self.drop(F.relu(self.bn1(self.gat1(h, edge_index))))
        return self.classifier(F.relu(self.bn2(self.gat2(h, edge_index))))

model = CMAGNN().to(DEVICE)
model.load_state_dict(torch.load(os.path.join(OUTPUTS_DIR, 'best_model.pth'), map_location=DEVICE))
model.eval()

scaler = joblib.load(os.path.join(PROCESSED_DIR, 'feature_scaler.pkl'))
mask   = np.load(os.path.join(PROCESSED_DIR, 'valid_mask.npy'))
shape  = tuple(np.load(os.path.join(PROCESSED_DIR, 'spatial_shape.npy')))
print('✅ Model and scaler loaded.')

In [ ]:
# ── STEPS 27-28: Predict & Reconstruct Annual Maps ───────────

def predict_year(year):
    """
    Load aligned datacube for a given year, run inference,
    and reconstruct the classified raster.

    Returns:
        classified_map : np.ndarray shape (H, W) with class codes 1–5 (0 = background)
        profile        : rasterio profile for saving
    """
    cube_path = os.path.join(PROCESSED_DIR, f'datacube_{year}_aligned.tif')

    with rasterio.open(cube_path) as src:
        cube = src.read().astype(np.float32)   # (8, H, W)
        profile = src.profile.copy()

    H, W = shape

    # Extract valid pixels
    X_year = cube[:, mask].T   # (N_valid, 8)

    # Normalise using 2025 scaler
    X_norm = scaler.transform(X_year).astype(np.float32)

    # Reload graph edges for this year
    # (Use same spatial graph structure — edges don't change between years)
    graph_data = torch.load(os.path.join(PROCESSED_DIR, 'graph_data.pt'))
    edge_index  = graph_data.edge_index.to(DEVICE)

    x_tensor = torch.tensor(X_norm).to(DEVICE)

    # Inference in batches to manage memory
    CHUNK = 50_000
    all_preds = []
    # Note: for full graph, run on full graph with edge_index
    with torch.no_grad():
        preds = model(x_tensor, edge_index).argmax(dim=1).cpu().numpy()
    preds += 1   # back to 1-indexed class codes

    # Reconstruct raster
    classified_map = np.zeros((H, W), dtype=np.uint8)
    classified_map[mask] = preds

    return classified_map, profile


def save_classified_map(classified_map, profile, year):
    """Save classified raster as GeoTIFF."""
    out_path = os.path.join(EXPORTS_DIR, f'landcover_{year}.tif')
    profile.update({'count': 1, 'dtype': 'uint8', 'nodata': 0})
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(classified_map[np.newaxis, :, :])
    print(f'  ✅ Saved: landcover_{year}.tif')
    return out_path


# ── STEP 29: Generate Annual Maps ────────────────────────────
all_maps = {}
for year in YEARS:
    print(f'Processing {year}...')
    classified_map, profile = predict_year(year)
    save_classified_map(classified_map, profile, year)
    all_maps[year] = classified_map

print(f'\n✅ Annual maps saved for: {YEARS}')

In [ ]:
# ── Visualise Annual Maps (2x4 grid) ─────────────────────────

cmap   = mcolors.ListedColormap(['black'] + CLASS_COLORS)   # 0=background
bounds = [0, 1, 2, 3, 4, 5, 6]
norm   = mcolors.BoundaryNorm(bounds, cmap.N)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, year in enumerate(YEARS):
    ax = axes[i // 4][i % 4]
    ax.imshow(all_maps[year], cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(str(year), fontsize=12, fontweight='bold')
    ax.axis('off')

# Legend
patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=10,
           title='Landcover Class', title_fontsize=11)
plt.suptitle('Obuasi Municipality Landcover Maps 2018–2025', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig(os.path.join(OUTPUTS_DIR, 'annual_landcover_maps.png'), dpi=150)
plt.show()
print('✅ Annual maps figure saved.')